# Profiling Dataset Berita Politik Indonesia

## Dataset

Kumpulan berita politik dari portal berita Indonesia, diambil dari Hugging
Face. Satu barisnya satu berita, lengkap dengan judul, isi, penulis, portal,
waktu terbit, kata kunci, dan kota/kabupaten yang dibahas.

| | |
|---|---|
| Nama | Indonesian Political News Clean |
| Sumber | https://huggingface.co/datasets/ardimardiana/indonesian-political-news-clean |
| Baris | 1.200.000 |
| Kolom | 10 |
| Ukuran berkas | 968,4 MB |
| Periode | 2015 sampai 2026, terpusat di 2025 dan 2026 |

Dataset aslinya 22 berkas Parquet berisi 1.628.288 baris dan 15 kolom, totalnya
5,4 GB. `src/download_dataset.py` mengunduh berkas aslinya ke
`data/raw/berita_politik/` tanpa mengubah isinya, lalu menyusun tabel analisis
1.200.000 baris dengan 10 kolom ke `data/processed/`. Kolom `news_raw_data`
dibuang karena isinya JSON mentah yang menduplikasi `news_text`. Rinciannya ada
di `data/README.md`.

Syarat tugas minimal 500 MB atau lebih dari 1.000.000 baris. Tabel ini
memenuhi dua-duanya.

## Library

In [1]:
import sys
from pathlib import Path

import duckdb
import polars as pl

print("polars :", pl.__version__)
print("duckdb :", duckdb.__version__)

polars : 1.44.2
duckdb : 1.5.5


## Baca data

Path dihitung dari folder kerja, supaya notebook bisa dijalankan dari root repo
maupun dari dalam folder `notebooks/`. Modul di `src/` ikut dipakai supaya
definisi path dan nama kolomnya hanya ada di satu tempat.

In [2]:
akar = Path.cwd()
while not (akar / "data").is_dir() and akar != akar.parent:
    akar = akar.parent

sys.path.insert(0, str(akar))
from src import dataset as ds

DATA_PATH = ds.DATA_ANALISIS

print(DATA_PATH.relative_to(akar))
print(f"ukuran : {DATA_PATH.stat().st_size / 1e6:,.1f} MB")

data\processed\berita_politik_1_2jt.parquet
ukuran : 968.4 MB


## Skema kolom

Dibaca dengan `scan_parquet` (lazy), jadi isi berkas belum dimuat ke memori.

In [3]:
data = pl.scan_parquet(DATA_PATH)

for nama, tipe in data.collect_schema().items():
    print(f"{nama:15} {tipe}")

id              String
news_title      String
news_source     String
news_author     String
news_hostname   String
news_date       String
news_text       String
news_tags       String
news_kabkot     String
news_intent     String


In [4]:
jumlah_baris = data.select(pl.len()).collect().item()
jumlah_kolom = len(data.collect_schema())
ukuran_mb = DATA_PATH.stat().st_size / 1e6

print(f"baris  : {jumlah_baris:,}")
print(f"kolom  : {jumlah_kolom}")
print(f"ukuran : {ukuran_mb:,.1f} MB")
print()
print("syarat tugas (>1.000.000 baris) :", "terpenuhi" if jumlah_baris > 1_000_000 else "BELUM")
print("syarat tugas (>=500 MB)         :", "terpenuhi" if ukuran_mb >= 500 else "BELUM")

baris  : 1,200,000
kolom  : 10
ukuran : 968.4 MB

syarat tugas (>1.000.000 baris) : terpenuhi
syarat tugas (>=500 MB)         : terpenuhi


## Arti kolom

| Kolom | Isi |
|---|---|
| `id` | penanda unik berita |
| `news_title` | judul berita |
| `news_source` | alamat lengkap halaman berita |
| `news_author` | nama penulis |
| `news_hostname` | nama domain portal, mis. `disway.id` |
| `news_date` | waktu terbit, format `YYYY-MM-DD HH:MM:SS+00:00` |
| `news_text` | isi lengkap berita |
| `news_tags` | daftar kata kunci berita |
| `news_kabkot` | kabupaten atau kota yang dibahas |
| `news_intent` | hasil analisis maksud berita, memuat daftar tempat |

In [5]:
data.select(["id", "news_title", "news_hostname", "news_date", "news_kabkot"]).head(5).collect()

id,news_title,news_hostname,news_date,news_kabkot
str,str,str,str,str
"""1924973""","""Dana Kegiatan Isbat Nikah Baka…","""disway.id""","""2026-04-27 01:30:00+00:00""","""Kabupaten Mukomuko"""
"""1925951""","""Satu Orang Meninggal dari Insi…","""ayokaltim.com""","""2026-04-27 03:19:46+00:00""","""Kota Samarinda"""
"""1925952""","""DPRD Bongkar Ketidaksesuaian L…","""infosatu.co""","""2026-04-27 03:59:29+00:00""","""Kota Samarinda"""
"""1925953""","""DPRD Dorong Penanganan Banjir …","""infosatu.co""","""2026-04-27 01:43:24+00:00""","""Kota Samarinda"""
"""1924974""","""Kepung Pasar Karang Nabire, Ma…","""tribunnews.com""","""2026-04-27 02:14:39+00:00""","""Kabupaten Nabire"""


## Cek nilai kosong

In [6]:
kolom = data.collect_schema().names()
null_per_kolom = data.select([pl.col(k).null_count().alias(k) for k in kolom]).collect().row(0)

for k, n in zip(kolom, null_per_kolom):
    print(f"{k:15} {n:>10,}  ({n / jumlah_baris * 100:5.2f}%)")

id                       0  ( 0.00%)
news_title               0  ( 0.00%)
news_source              0  ( 0.00%)
news_author         94,607  ( 7.88%)
news_hostname            0  ( 0.00%)
news_date           60,333  ( 5.03%)
news_text                0  ( 0.00%)
news_tags                0  ( 0.00%)
news_kabkot              0  ( 0.00%)
news_intent              0  ( 0.00%)


Yang kosong cuma `news_author` dan `news_date`. Delapan kolom lainnya bersih.

## Duplikat

In [7]:
data.select([
    pl.col("id").n_unique().alias("id_unik"),
    pl.col("news_title").n_unique().alias("judul_unik"),
    pl.len().alias("jumlah_baris"),
]).collect()

id_unik,judul_unik,jumlah_baris
u32,u32,u32
1200000,1200000,1200000


## Rentang tanggal

Kolom `news_date` masih teks, diubah ke datetime dulu. Setelah itu langsung
terlihat ada nilai yang bentuknya benar tapi isinya tidak masuk akal.

In [8]:
data_tanggal = data.with_columns(ds.waktu_terbit().alias("tgl"))

data_tanggal.select([
    pl.col("tgl").min().alias("terbit_awal"),
    pl.col("tgl").max().alias("terbit_akhir"),
    pl.col("tgl").dt.year().n_unique().alias("jumlah_tahun"),
]).collect()

terbit_awal,terbit_akhir,jumlah_tahun
"datetime[μs, UTC]","datetime[μs, UTC]",u32
0001-11-30 00:00:00 UTC,2285-11-30 17:00:00 UTC,48


In [9]:
per_tahun = (
    data_tanggal.with_columns(pl.col("tgl").dt.year().alias("tahun"))
    .group_by("tahun")
    .agg(pl.len().alias("jumlah_berita"))
    .sort("jumlah_berita", descending=True)
    .head(8)
    .collect()
)
per_tahun

tahun,jumlah_berita
i32,u32
2026,753730
2025,383532
null,60333
2018,586
2020,348
2024,298
2023,214
2022,186


Tahun 0001 dan 2285 jelas salah, tapi tidak terbaca sebagai error saat data
dibuka. Nilai seperti ini harus disaring pakai rentang tanggal yang wajar di
notebook pembersihan.

In [10]:
data_tanggal.select([
    pl.col("tgl").dt.year().is_null().sum().alias("kosong_atau_gagal_dibaca"),
    ((pl.col("tgl").dt.year() < 2015) | (pl.col("tgl").dt.year() > 2026)).sum().alias("di_luar_2015_2026"),
    ((pl.col("tgl").dt.year() >= 2015) & (pl.col("tgl").dt.year() <= 2026)).sum().alias("wajar_2015_2026"),
]).collect()

kosong_atau_gagal_dibaca,di_luar_2015_2026,wajar_2015_2026
u32,u32,u32
60333,284,1139383


## Cakupan wilayah

In [11]:
data.group_by("news_kabkot").agg(pl.len().alias("jumlah_berita")).sort("jumlah_berita", descending=True).head(10).collect()

news_kabkot,jumlah_berita
str,u32
"""Kota Malang""",11924
"""Kota Bandung""",11211
"""Kota Medan""",10829
"""Kota Semarang""",10176
"""Kota Bogor""",9713
"""Kota Surabaya""",9129
"""Kota Tangerang""",9052
"""Kota Makassar""",8883
"""Kota Padang""",8240


In [12]:
data.select(pl.col("news_kabkot").n_unique().alias("jumlah_kabkot")).collect()

jumlah_kabkot
u32
1003


## Portal berita

In [13]:
data.group_by("news_hostname").agg(pl.len().alias("jumlah_berita")).sort("jumlah_berita", descending=True).head(10).collect()

news_hostname,jumlah_berita
str,u32
"""tribunnews.com""",81711
"""rri.co.id""",72748
"""jawapos.com""",29165
"""pikiran-rakyat.com""",25777
"""antaranews.com""",24550
"""detik.com""",22576
"""disway.id""",21014
"""kompas.com""",12279
"""merdeka.com""",9683


In [14]:
data.select(pl.col("news_hostname").n_unique().alias("jumlah_portal")).collect()

jumlah_portal
u32
5336


## SUMMARIZE (DuckDB)

DuckDB bisa membaca berkas Parquet langsung, tanpa mengimpor datanya dulu.
`SUMMARIZE` mengeluarkan statistik tiap kolom dalam satu perintah.

In [15]:
con = duckdb.connect()
con.execute("SET threads = 4")

profil = con.execute(f"""
    SUMMARIZE SELECT
        id, news_author, news_hostname, news_date, news_kabkot
    FROM read_parquet('{DATA_PATH.as_posix()}')
""").pl()

profil.select(["column_name", "min", "max", "approx_unique", "count", "null_percentage"])

column_name,min,max,approx_unique,count,null_percentage
str,str,str,i64,i64,"decimal[9,2]"
"""id""","""1""","""999999""",1315010,1200000,0.00
"""news_author""","""( )""","""𝗗𝟯𝗹 𝟬𝟮""",50594,1200000,7.88
"""news_hostname""","""1kabar.com""","""zuper.fajar.co.id""",5196,1200000,0.00
"""news_date""","""0001-11-30 00:00:00+00:00""","""2285-11-30 17:00:00+00:00""",713370,1200000,5.03
"""news_kabkot""","""Aceh Barat""","""Yalimo""",1044,1200000,0.00


## Panjang teks

Panjang judul dan isi berita dipakai untuk mencari nilai yang menyimpang.

In [16]:
data_teks = data.with_columns(ds.panjang_judul(), ds.panjang_teks())

data_teks.select([
    pl.col("panjang_judul").min().alias("judul_min"),
    pl.col("panjang_judul").median().alias("judul_median"),
    pl.col("panjang_judul").max().alias("judul_maks"),
    pl.col("panjang_teks").min().alias("teks_min"),
    pl.col("panjang_teks").median().alias("teks_median"),
    pl.col("panjang_teks").max().alias("teks_maks"),
]).collect()

judul_min,judul_median,judul_maks,teks_min,teks_median,teks_maks
u32,f64,u32,u32,f64,u32
1,86.0,4310,143,1830.0,1062327


In [17]:
data_teks.select([
    (pl.col("panjang_teks") < 300).sum().alias("teks_di_bawah_300"),
    (pl.col("panjang_teks") > 50000).sum().alias("teks_di_atas_50rb"),
    (pl.col("panjang_judul") < 10).sum().alias("judul_di_bawah_10"),
]).collect()

teks_di_bawah_300,teks_di_atas_50rb,judul_di_bawah_10
u32,u32,u32
67557,7,90


## Catatan

- 1.200.000 baris, 10 kolom, 968,4 MB. Syarat tugas terpenuhi dari dua sisi.
- Yang kosong cuma `news_author` (94.607 baris, 7,88%) dan `news_date` (60.333 baris, 5,03%). Delapan kolom lain bersih.
- `id` dan `news_title` semuanya unik, tidak ada duplikat baris.
- Tanggal terbit mayoritas 2025 dan 2026. Tapi ada 284 baris di luar 2015-2026, dan tahunnya ada yang 0001 dan 2285. Ini nilai rusak yang harus disaring pakai rentang wajar.
- Ada 60.333 baris yang tanggalnya kosong. Kalau dianalisis per waktu, baris ini harus dikecualikan dulu.
- Wilayahnya tersebar luas. Kota Malang paling banyak (11.924 berita), disusul Kota Bandung (11.211) dan Kota Medan (10.829).
- Portal terbanyak `tribunnews.com` (81.711) dan `rri.co.id` (72.748). Datanya datang dari banyak portal, jadi sebarannya tidak bergantung satu sumber.
- Panjang judul 1 sampai 4.310 karakter, median 86. Panjang isi 143 sampai 1.062.327 karakter, median 1.830.
- Ada 67.557 baris (5,63%) dengan isi di bawah 300 karakter, dan 7 baris di atas 50.000 karakter. Dua-duanya kandidat buangan atau perlu penanganan khusus di tahap pembersihan.
- Isi berita terpanjang 1.062.327 karakter, jauh di atas median. Kemungkinan isinya gabungan beberapa halaman, perlu diperiksa satu per satu.